# Quarter-Car Suspension Dynamics & Filtering
Quick exploratory analysis of vehicle response across speeds and Butterworth filter performance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
import sys
sys.path.append("..")
from src.dynamics import quarter_car_ode, PARAMS
from src.filter import apply_lowpass_filter

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

## 1. Simulation Across Speeds (36 km/h vs 72 km/h)

In [ ]:
distance = np.linspace(0, 50, 1000)
elevation = 0.05 * np.exp(-((distance - 10.0)**2) / 1.0)
road_interp = interp1d(distance, elevation, bounds_error=False, fill_value=0.0)

def run_sim(v_car):
    p = PARAMS.copy()
    p['v_car'] = v_car
    t_max = distance[-1] / v_car
    t_eval = np.linspace(0, t_max, len(distance))
    
    sol = solve_ivp(lambda t, y: quarter_car_ode(t, y, road_interp, p), (0, t_max), [0,0,0,0], t_eval=t_eval, method='RK45')
    a_s = [(p['k_s']*(sol.y[2][i]-sol.y[0][i]) + p['c_s']*(sol.y[3][i]-sol.y[1][i])) / p['m_s'] for i in range(len(sol.t))]
    
    return pd.DataFrame({'time_s': sol.t, 'disp_cm': sol.y[0] * 100, 'accel_ms2': a_s})

df_low, df_high = run_sim(10.0), run_sim(20.0)

# Plot Responses
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
ax1.plot(df_low['time_s'], df_low['disp_cm'], label='36 km/h', color='tab:blue')
ax1.plot(df_high['time_s'], df_high['disp_cm'], label='72 km/h', color='tab:red')
ax1.set_ylabel("Displacement (cm)"); ax1.legend(); ax1.grid(True, linestyle=':')

ax2.plot(df_low['time_s'], df_low['accel_ms2'], label='36 km/h', color='tab:blue')
ax2.plot(df_high['time_s'], df_high['accel_ms2'], label='72 km/h', color='tab:red')
ax2.set_xlabel("Time (s)"); ax2.set_ylabel("Acceleration ($m/s^2$)"); ax2.legend(); ax2.grid(True, linestyle=':')
plt.tight_layout(); plt.show()

## 2. Low-Pass Filter Performance

In [ ]:
np.random.seed(42)
noisy = df_high['accel_ms2'] + np.random.normal(0, 0.3, len(df_high))
fs = 1.0 / np.mean(np.diff(df_high['time_s']))
filtered = apply_lowpass_filter(noisy, cutoff_hz=15.0, fs=fs)

plt.figure(figsize=(10, 3))
plt.plot(df_high['time_s'], noisy, label='Noisy Sensor', alpha=0.4, color='gray')
plt.plot(df_high['time_s'], filtered, label='Filtered (15 Hz)', color='tab:green', linewidth=2)
plt.xlabel("Time (s)"); plt.ylabel("Acceleration ($m/s^2$)"); plt.legend(); plt.grid(True, linestyle=':')
plt.show()